# Retail Sales Forecasting and Customer Analytics

## 01 - Raw Data Inspection

This notebook is used to inspect the original Olist Brazilian E-Commerce datasets before data cleaning and transformation. The purpose is to understand the structure of the datasets, identify data quality issues and determine the preparation required for further analysis.

In [4]:
import pandas as pd

In [1]:
from pathlib import Path

raw_data_path = Path("../01_Raw_Data")

print(raw_data_path)

..\01_Raw_Data


In [2]:
csv_files = list(raw_data_path.glob("*.csv"))

for file in csv_files:
    print(file.name)

olist_customers_dataset.csv
olist_geolocation_dataset.csv
olist_orders_dataset.csv
olist_order_items_dataset.csv
olist_order_payments_dataset.csv
olist_order_reviews_dataset.csv
olist_products_dataset.csv
olist_sellers_dataset.csv
product_category_name_translation.csv


## Loading the Raw Datasets

The nine Olist datasets are loaded into separate pandas DataFrames. Keeping the datasets separate at this stage allows their structure and relationships to be examined before any cleaning, transformation or merging is performed.

In [5]:
customers = pd.read_csv(raw_data_path / "olist_customers_dataset.csv")
geolocation = pd.read_csv(raw_data_path / "olist_geolocation_dataset.csv")
order_items = pd.read_csv(raw_data_path / "olist_order_items_dataset.csv")
payments = pd.read_csv(raw_data_path / "olist_order_payments_dataset.csv")
reviews = pd.read_csv(raw_data_path / "olist_order_reviews_dataset.csv")
orders = pd.read_csv(raw_data_path / "olist_orders_dataset.csv")
products = pd.read_csv(raw_data_path / "olist_products_dataset.csv")
sellers = pd.read_csv(raw_data_path / "olist_sellers_dataset.csv")
category_translation = pd.read_csv(
    raw_data_path / "product_category_name_translation.csv"
)

In [6]:
## Check the size of each dataset
datasets = {
    "Customers": customers,
    "Geolocation": geolocation,
    "Order Items": order_items,
    "Payments": payments,
    "Reviews": reviews,
    "Orders": orders,
    "Products": products,
    "Sellers": sellers,
    "Category Translation": category_translation
}

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows, {df.shape[1]} columns")

Customers: 99,441 rows, 5 columns
Geolocation: 1,000,163 rows, 5 columns
Order Items: 112,650 rows, 7 columns
Payments: 103,886 rows, 5 columns
Reviews: 99,224 rows, 7 columns
Orders: 99,441 rows, 8 columns
Products: 32,951 rows, 9 columns
Sellers: 3,095 rows, 4 columns
Category Translation: 71 rows, 2 columns


In [7]:
## Inspect the columns
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(df.columns.tolist())


Customers
--------------------------------------------------
['customer_id', 'customer_unique_id', 'customer_zip_code_prefix', 'customer_city', 'customer_state']

Geolocation
--------------------------------------------------
['geolocation_zip_code_prefix', 'geolocation_lat', 'geolocation_lng', 'geolocation_city', 'geolocation_state']

Order Items
--------------------------------------------------
['order_id', 'order_item_id', 'product_id', 'seller_id', 'shipping_limit_date', 'price', 'freight_value']

Payments
--------------------------------------------------
['order_id', 'payment_sequential', 'payment_type', 'payment_installments', 'payment_value']

Reviews
--------------------------------------------------
['review_id', 'order_id', 'review_score', 'review_comment_title', 'review_comment_message', 'review_creation_date', 'review_answer_timestamp']

Orders
--------------------------------------------------
['order_id', 'customer_id', 'order_status', 'order_purchase_timestamp', 'orde

## Initial Data Quality Assessment

The structure, data types, missing values and duplicate records of each dataset are examined before any cleaning decisions are made.

In [8]:
## Check the data types of each dataset
for name, df in datasets.items():
    print(f"\n{name}")
    print("-" * 50)
    print(df.dtypes)


Customers
--------------------------------------------------
customer_id                   str
customer_unique_id            str
customer_zip_code_prefix    int64
customer_city                 str
customer_state                str
dtype: object

Geolocation
--------------------------------------------------
geolocation_zip_code_prefix      int64
geolocation_lat                float64
geolocation_lng                float64
geolocation_city                   str
geolocation_state                  str
dtype: object

Order Items
--------------------------------------------------
order_id                   str
order_item_id            int64
product_id                 str
seller_id                  str
shipping_limit_date        str
price                  float64
freight_value          float64
dtype: object

Payments
--------------------------------------------------
order_id                    str
payment_sequential        int64
payment_type                str
payment_installments      int

In [10]:
## Check for missing values in each dataset
for name, df in datasets.items():
    missing = df.isnull().sum()
    missing = missing[missing > 0]

    print(f"\n{name}")
    print("-" * 50)

    if len(missing) == 0:
        print("No missing values")
    else:
        print(missing)


Customers
--------------------------------------------------
No missing values

Geolocation
--------------------------------------------------
No missing values

Order Items
--------------------------------------------------
No missing values

Payments
--------------------------------------------------
No missing values

Reviews
--------------------------------------------------
review_comment_title      87656
review_comment_message    58247
dtype: int64

Orders
--------------------------------------------------
order_approved_at                 160
order_delivered_carrier_date     1783
order_delivered_customer_date    2965
dtype: int64

Products
--------------------------------------------------
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                2
product_length_cm               2
product_height_cm               2
product_width_cm                2
dtype: int64

Sellers

In [11]:
## Check the duplicate rows in each dataset
for name, df in datasets.items():
    duplicate_count = df.duplicated().sum()
    print(f"{name}: {duplicate_count:,} duplicate rows")

Customers: 0 duplicate rows
Geolocation: 261,831 duplicate rows
Order Items: 0 duplicate rows
Payments: 0 duplicate rows
Reviews: 0 duplicate rows
Orders: 0 duplicate rows
Products: 0 duplicate rows
Sellers: 0 duplicate rows
Category Translation: 0 duplicate rows


In [12]:
## Investigate the missing order dates
print("Order Status Counts")
print("-" * 50)
print(orders["order_status"].value_counts())

print("\nMissing Approval Date by Order Status")
print("-" * 50)
print(
    orders.loc[orders["order_approved_at"].isna(), "order_status"]
    .value_counts()
)

print("\nMissing Carrier Delivery Date by Order Status")
print("-" * 50)
print(
    orders.loc[orders["order_delivered_carrier_date"].isna(), "order_status"]
    .value_counts()
)

print("\nMissing Customer Delivery Date by Order Status")
print("-" * 50)
print(
    orders.loc[orders["order_delivered_customer_date"].isna(), "order_status"]
    .value_counts()
)

Order Status Counts
--------------------------------------------------
order_status
delivered      96478
shipped         1107
canceled         625
unavailable      609
invoiced         314
processing       301
created            5
approved           2
Name: count, dtype: int64

Missing Approval Date by Order Status
--------------------------------------------------
order_status
canceled     141
delivered     14
created        5
Name: count, dtype: int64

Missing Carrier Delivery Date by Order Status
--------------------------------------------------
order_status
unavailable    609
canceled       550
invoiced       314
processing     301
created          5
approved         2
delivered        2
Name: count, dtype: int64

Missing Customer Delivery Date by Order Status
--------------------------------------------------
order_status
shipped        1107
canceled        619
unavailable     609
invoiced        314
processing      301
delivered         8
created           5
approved          2


In [13]:
## Investigate the product missing values
missing_product_info = products[
    products["product_category_name"].isna()
]

print("Products with missing category:", len(missing_product_info))

print("\nMissing values within these products:")
print(missing_product_info.isnull().sum())

Products with missing category: 610

Missing values within these products:
product_id                      0
product_category_name         610
product_name_lenght           610
product_description_lenght    610
product_photos_qty            610
product_weight_g                1
product_length_cm               1
product_height_cm               1
product_width_cm                1
dtype: int64


In [14]:
print("Products with missing physical measurements:")
print(
    products[
        products[
            [
                "product_weight_g",
                "product_length_cm",
                "product_height_cm",
                "product_width_cm"
            ]
        ].isnull().any(axis=1)
    ]
)

Products with missing physical measurements:
                             product_id product_category_name  \
8578   09ff539a621711667c43eba6a3bd8466                 bebes   
18851  5eb564652db742ff8f28759cd8d2652a                   NaN   

       product_name_lenght  product_description_lenght  product_photos_qty  \
8578                  60.0                       865.0                 3.0   
18851                  NaN                         NaN                 NaN   

       product_weight_g  product_length_cm  product_height_cm  \
8578                NaN                NaN                NaN   
18851               NaN                NaN                NaN   

       product_width_cm  
8578                NaN  
18851               NaN  


In [15]:
## Investigate the Geolocation duplicates
print("Total geolocation rows:", len(geolocation))
print("Completely duplicated rows:", geolocation.duplicated().sum())
print("Unique complete rows:", len(geolocation.drop_duplicates()))

print("\nUnique postcode prefixes:")
print(geolocation["geolocation_zip_code_prefix"].nunique())

print("\nNumber of records per postcode prefix:")
print(geolocation["geolocation_zip_code_prefix"].value_counts().describe())

Total geolocation rows: 1000163
Completely duplicated rows: 261831
Unique complete rows: 738332

Unique postcode prefixes:
19015

Number of records per postcode prefix:
count    19015.000000
mean        52.598633
std         72.057907
min          1.000000
25%         10.000000
50%         29.000000
75%         66.500000
max       1146.000000
Name: count, dtype: float64


In [16]:
## Check key uniqueness and relationships
print("CUSTOMERS")
print("-" * 50)
print("Customer rows:", len(customers))
print("Unique customer_id:", customers["customer_id"].nunique())
print("Unique customer_unique_id:", customers["customer_unique_id"].nunique())

print("\nORDERS")
print("-" * 50)
print("Order rows:", len(orders))
print("Unique order_id:", orders["order_id"].nunique())
print("Unique customer_id in orders:", orders["customer_id"].nunique())

print("\nORDER ITEMS")
print("-" * 50)
print("Order item rows:", len(order_items))
print("Unique order_id:", order_items["order_id"].nunique())
print("Unique product_id:", order_items["product_id"].nunique())
print("Unique seller_id:", order_items["seller_id"].nunique())

print("\nPRODUCTS")
print("-" * 50)
print("Product rows:", len(products))
print("Unique product_id:", products["product_id"].nunique())

print("\nSELLERS")
print("-" * 50)
print("Seller rows:", len(sellers))
print("Unique seller_id:", sellers["seller_id"].nunique())

CUSTOMERS
--------------------------------------------------
Customer rows: 99441
Unique customer_id: 99441
Unique customer_unique_id: 96096

ORDERS
--------------------------------------------------
Order rows: 99441
Unique order_id: 99441
Unique customer_id in orders: 99441

ORDER ITEMS
--------------------------------------------------
Order item rows: 112650
Unique order_id: 98666
Unique product_id: 32951
Unique seller_id: 3095

PRODUCTS
--------------------------------------------------
Product rows: 32951
Unique product_id: 32951

SELLERS
--------------------------------------------------
Seller rows: 3095
Unique seller_id: 3095


In [17]:
## Investigate repeat customer IDs
customer_id_counts = (
    customers.groupby("customer_unique_id")["customer_id"]
    .nunique()
)

print("Customer IDs per unique customer:")
print(customer_id_counts.value_counts().sort_index())

print("\nMaximum customer IDs linked to one unique customer:")
print(customer_id_counts.max())

print("\nUnique customers linked to more than one customer_id:")
print((customer_id_counts > 1).sum())

Customer IDs per unique customer:
customer_id
1     93099
2      2745
3       203
4        30
5         8
6         6
7         3
9         1
17        1
Name: count, dtype: int64

Maximum customer IDs linked to one unique customer:
17

Unique customers linked to more than one customer_id:
2997


In [18]:
## Check the raw date range
purchase_dates = pd.to_datetime(
    orders["order_purchase_timestamp"],
    errors="coerce"
)

print("Earliest order purchase:")
print(purchase_dates.min())

print("\nLatest order purchase:")
print(purchase_dates.max())

print("\nInvalid purchase timestamps:")
print(purchase_dates.isna().sum())

Earliest order purchase:
2016-09-04 21:15:19

Latest order purchase:
2018-10-17 17:30:18

Invalid purchase timestamps:
0


In [19]:
## Check order activity by month
monthly_orders = (
    purchase_dates
    .dt.to_period("M")
    .value_counts()
    .sort_index()
)

print("Orders by month")
print("-" * 50)
print(monthly_orders)

Orders by month
--------------------------------------------------
order_purchase_timestamp
2016-09       4
2016-10     324
2016-12       1
2017-01     800
2017-02    1780
2017-03    2682
2017-04    2404
2017-05    3700
2017-06    3245
2017-07    4026
2017-08    4331
2017-09    4285
2017-10    4631
2017-11    7544
2017-12    5673
2018-01    7269
2018-02    6728
2018-03    7211
2018-04    6939
2018-05    6873
2018-06    6167
2018-07    6292
2018-08    6512
2018-09      16
2018-10       4
Freq: M, Name: count, dtype: int64


In [20]:
## Check whether Orders and Order Items match
orders_without_items = orders[
    ~orders["order_id"].isin(order_items["order_id"])
]

print("Orders without order items:")
print(len(orders_without_items))

print("\nOrder status of orders without items:")
print(orders_without_items["order_status"].value_counts())

Orders without order items:
775

Order status of orders without items:
order_status
unavailable    603
canceled       164
created          5
invoiced         2
shipped          1
Name: count, dtype: int64


In [21]:
## Check Payments and Reviews relationships
print("PAYMENTS")
print("-" * 50)
print("Payment rows:", len(payments))
print("Unique order IDs in payments:", payments["order_id"].nunique())

orders_without_payments = orders[
    ~orders["order_id"].isin(payments["order_id"])
]

print("Orders without payment records:", len(orders_without_payments))

print("\nREVIEWS")
print("-" * 50)
print("Review rows:", len(reviews))
print("Unique order IDs in reviews:", reviews["order_id"].nunique())

orders_without_reviews = orders[
    ~orders["order_id"].isin(reviews["order_id"])
]

print("Orders without review records:", len(orders_without_reviews))

PAYMENTS
--------------------------------------------------
Payment rows: 103886
Unique order IDs in payments: 99440
Orders without payment records: 1

REVIEWS
--------------------------------------------------
Review rows: 99224
Unique order IDs in reviews: 98673
Orders without review records: 768


## Initial Data Quality Findings

The initial inspection identified several issues that will need to be considered during data preparation:

- The datasets contain 99,441 orders and 96,096 unique customers based on `customer_unique_id`. A total of 2,997 unique customers are associated with more than one `customer_id`, which will need to be considered when carrying out customer-level analysis.
- Date and timestamp fields were initially imported as strings and will need to be converted to appropriate datetime formats before time-based analysis and forecasting.
- Missing values are concentrated in the Reviews, Orders and Products datasets. Review comments contain substantial missing data, while missing order dates are mainly associated with orders that were not completed. A small number of delivered orders also contain missing timeline information and require further consideration.
- A total of 610 products have missing category and descriptive information. Two products have missing physical measurement information.
- The Geolocation dataset contains 261,831 completely duplicated rows. Postcode prefixes also occur multiple times, meaning the table should not be directly joined to customer data without appropriate preparation.
- No complete duplicate rows were identified in the other eight datasets.
- The raw order purchase dates range from September 2016 to October 2018. However, the beginning and end of this period contain very few orders. The main continuous period of substantial activity is from January 2017 to August 2018, which will need to be considered when preparing data for forecasting.
- There are 775 orders without corresponding order-item records. Most of these are unavailable or cancelled orders, and no delivered orders were found without order-item records.
- One order has no payment record and 768 orders have no review record. Payments can contain multiple records per order, so the relationship between payment and order-item data must be handled carefully when datasets are joined.

No cleaning or removal decisions have been applied at this stage. These findings will be used to guide the data preparation process and justify the treatment of data-quality issues.